# 1 つの文書に複数のラベルを付ける（マルチラベル分類）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_classification/multilabel.ipynb)

NLP クックブック 12-10 ／ テキスト分類

1 つの記事が「経済」かつ「国際」のように、複数のカテゴリに同時に属する場合の分類をしたい。

In [1]:
import nltk
nltk.download("reuters", quiet=True)
from nltk.corpus import reuters
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, hamming_loss
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.svm import LinearSVC

train_ids = [i for i in reuters.fileids() if i.startswith("training/")]
test_ids = [i for i in reuters.fileids() if i.startswith("test/")]
print(len(train_ids), len(test_ids), "カテゴリ数:", len(reuters.categories()))
print(reuters.categories(train_ids[1]), reuters.raw(train_ids[1])[:80].replace("\n", " "))

mlb = MultiLabelBinarizer()
Y_train = mlb.fit_transform([reuters.categories(i) for i in train_ids])   # 0/1 の行列 (文書数 × ラベル数)
Y_test = mlb.transform([reuters.categories(i) for i in test_ids])
vec = TfidfVectorizer(stop_words="english", sublinear_tf=True, min_df=2)
X_train, X_test = vec.fit_transform(reuters.raw(i) for i in train_ids), vec.transform(reuters.raw(i) for i in test_ids)

7769 3019 カテゴリ数: 90
['acq'] COMPUTER TERMINAL SYSTEMS &lt;CPML> COMPLETES SALE   Computer Terminal Systems I


In [2]:
clf = OneVsRestClassifier(LinearSVC(C=1)).fit(X_train, Y_train)   # ラベルごとに 2 値分類器を学習
pred = clf.predict(X_test)
print("micro-F1:", round(f1_score(Y_test, pred, average="micro"), 3),
      " macro-F1:", round(f1_score(Y_test, pred, average="macro", zero_division=0), 3),
      " samples-F1:", round(f1_score(Y_test, pred, average="samples", zero_division=0), 3),
      " hamming_loss:", round(hamming_loss(Y_test, pred), 4),
      " 完全一致率:", round((pred == Y_test).all(axis=1).mean(), 3))
for i in range(3):
    print("正解:", mlb.inverse_transform(Y_test[i:i + 1])[0], " 予測:", mlb.inverse_transform(pred[i:i + 1])[0])

micro-F1: 0.862  macro-F1: 0.41  samples-F1: 0.859  hamming_loss: 0.0035  完全一致率: 0.808
正解: ('trade',)  予測: ('trade',)
正解: ('grain',)  予測: ('grain',)
正解: ('crude', 'nat-gas')  予測: ()


In [3]:
# 訓練データの 3 分割交差検証で予測確率を求め、学習側だけでしきい値を選ぶ
import warnings
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

def predict_with_fallback(proba, threshold):
    """しきい値以上のラベルを選び、全ラベルが未満の文書には最大確率の 1 ラベルを割り当てる"""
    pred = (proba >= threshold).astype(int)
    empty = pred.sum(axis=1) == 0
    if empty.any():
        pred[empty, proba[empty].argmax(axis=1)] = 1
    return pred

lr = OneVsRestClassifier(LogisticRegression(C=10, solver="liblinear"))
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)   # 分割の片側で正例が 0 件になる稀なラベルの警告を抑える
    proba_cv = cross_val_predict(lr, X_train, Y_train, cv=3, method="predict_proba")

thresholds = np.round(np.arange(0.15, 0.56, 0.05), 2)
best_th = max(thresholds, key=lambda th: f1_score(Y_train, predict_with_fallback(proba_cv, th), average="micro"))
print(f"CV で選んだしきい値: {best_th:.2f} (CV micro-F1 = {f1_score(Y_train, predict_with_fallback(proba_cv, best_th), average='micro'):.3f})")

lr.fit(X_train, Y_train)
proba_test = lr.predict_proba(X_test)
for name, pred_lr in [("既定 (0.50)", (proba_test >= 0.50).astype(int)),
                      (f"CV 選択 ({best_th:.2f})", predict_with_fallback(proba_test, best_th))]:
    print(f"{name:14} micro-F1: {f1_score(Y_test, pred_lr, average='micro'):.3f}"
          f"  macro-F1: {f1_score(Y_test, pred_lr, average='macro', zero_division=0):.3f}"
          f"  samples-F1: {f1_score(Y_test, pred_lr, average='samples', zero_division=0):.3f}"
          f"  hamming_loss: {hamming_loss(Y_test, pred_lr):.4f}"
          f"  完全一致率: {(pred_lr == Y_test).all(axis=1).mean():.3f}"
          f"  空予測: {int((pred_lr.sum(axis=1) == 0).sum())} 件")

CV で選んだしきい値: 0.25 (CV micro-F1 = 0.869)
既定 (0.50)      micro-F1: 0.832  macro-F1: 0.320  samples-F1: 0.811  hamming_loss: 0.0041  完全一致率: 0.765  空予測: 427 件
CV 選択 (0.25)   micro-F1: 0.873  macro-F1: 0.447  samples-F1: 0.905  hamming_loss: 0.0034  完全一致率: 0.829  空予測: 0 件


- micro-F1 は頻出ラベルに、macro-F1 は稀なラベルに引っ張られる。両者の差が大きいのは、稀なラベルが当たっていない証拠。
- `Samples-F1` (`average="samples"`) は各文書の正解集合と予測集合の F1 を文書全体で平均した値で、文書単位の取りこぼしが見えやすい。`Hamming Loss` (`hamming_loss`) は全「文書 × ラベル」のうち誤った割合を表し、小さいほど良い（ラベル数に対して正解ラベルが少ないデータでは 0 に近い小さな値になるため、F1 や完全一致率とあわせて比べる）。
- ラベル数が多いマルチラベル分類では、既定のしきい値 `0.50` のままだとどのラベルも基準を超えない空予測が増えやすい。訓練データの交差検証 (CV) で得た予測確率から学習側だけでしきい値を選び、すべて未満になった文書に最大確率の 1 ラベルを割り当てるフォールバックを入れると、空予測を防いで性能を改善できる（テストデータを見てしきい値を決めないこと）。
- BERT でやる場合は `AutoModelForSequenceClassification(..., problem_type="multi_label_classification")` とし、ラベルを float の 0/1 ベクトルで与える（損失が BCE になる）。